# ORION wake word training - openWakeWord Colab

Notebook reproducible para entrenar `orion_v2.onnx` con openWakeWord en Google Colab.

Runtime recomendado: Google Colab 2025.07, Python 3.11, GPU T4.

Todo el estado persistente vive en Google Drive:

```text
/content/drive/MyDrive/orion_wakeword_training/
```

Este notebook no conecta el wake word con ORION, no toca Whisper, Ollama, Policy Engine, ExecutionService ni `main.py`, no usa AudioSet y no convierte a TFLite.


In [ ]:
# 1. Montar Drive y definir rutas persistentes
from google.colab import drive
from pathlib import Path
import os

DRIVE_ROOT = Path("/content/drive/MyDrive/orion_wakeword_training")
drive.mount("/content/drive")

OPENWAKEWORD_DIR = Path("/content/openWakeWord")
PIPER_DIR = Path("/content/piper-sample-generator-v2")
DATASETS_DIR = DRIVE_ROOT / "datasets"
FMA_PATH = DATASETS_DIR / "fma"
MIT_RIRS_PATH = DATASETS_DIR / "mit_rirs"
FEATURES_DIR = DRIVE_ROOT / "features"
PIPER_MODELS_DRIVE = DRIVE_ROOT / "models" / "piper"
OUTPUT_DIR = DRIVE_ROOT / "training_output" / "orion_v2"
EXPORTS_DIR = DRIVE_ROOT / "exports"
CONFIG_PATH = DRIVE_ROOT / "my_model_v2.yaml"
AUGMENT_LOG = DRIVE_ROOT / "augment_clips.log"
TRAIN_LOG = DRIVE_ROOT / "train_model.log"

for path in [DRIVE_ROOT, DATASETS_DIR, FMA_PATH, MIT_RIRS_PATH, FEATURES_DIR, PIPER_MODELS_DRIVE, OUTPUT_DIR, EXPORTS_DIR]:
    path.mkdir(parents=True, exist_ok=True)

print(f"Drive persistente: {DRIVE_ROOT}")
print("Piper temporal:", PIPER_DIR)
print("openWakeWord temporal:", OPENWAKEWORD_DIR)


In [ ]:
# 2. Verificar GPU y Python
import platform
import subprocess
import sys

print("Python:", sys.version)
if sys.version_info[:2] != (3, 11):
    raise RuntimeError("Este notebook esta preparado para Python 3.11. Cambia el runtime de Colab antes de continuar.")

try:
    gpu_info = subprocess.check_output(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"], text=True).strip()
except Exception as exc:
    raise RuntimeError("No se detecto GPU. Activa GPU T4 en Runtime > Change runtime type.") from exc

print("GPU:", gpu_info)
if "T4" not in gpu_info:
    print("Aviso: el runtime recomendado es GPU T4. Puedes continuar si sabes que esta GPU es compatible.")


In [ ]:
# 3. Instalar dependencias una sola vez
import importlib.metadata
import subprocess
import sys
from pathlib import Path

packages = [
    "piper-phonemize==1.1.0",
    "webrtcvad",
    "torchinfo==1.8.0",
    "torchmetrics==1.2.0",
    "lightning-utilities",
    "speechbrain==0.5.14",
    "audiomentations==0.33.0",
    "torch-audiomentations==0.11.0",
    "acoustics==0.2.6",
    "pronouncing==0.2.0",
    "deep-phonemizer==0.0.19",
    "mutagen==1.47.0",
    "librosa==0.10.2.post1",
    "soxr==0.5.0.post1",
    "onnxruntime==1.27.0",
    "onnx==1.16.1",
    "ai-edge-litert==2.1.5",
    "speexdsp-ns==0.1.2",
    "fsspec==2023.6.0",
    "datasets==2.14.4",
]

constraints = Path("/tmp/orion_pip_constraints.txt")
try:
    hub_version = importlib.metadata.version("huggingface_hub")
except importlib.metadata.PackageNotFoundError:
    hub_version = None

constraints.write_text(f"huggingface_hub=={hub_version}
" if hub_version else "", encoding="utf-8")
print("huggingface_hub fijado durante pip:" if hub_version else "huggingface_hub no estaba instalado antes de pip", hub_version or "")

cmd = [sys.executable, "-m", "pip", "install", "-q", "-c", str(constraints), *packages]
print("Instalando dependencias necesarias. No se instala tensorflow-cpu==2.8.1.")
subprocess.check_call(cmd)
print("Dependencias listas.")


In [ ]:
# 4. Preparar openWakeWord temporal
import os
import subprocess
from pathlib import Path

if not OPENWAKEWORD_DIR.exists():
    subprocess.check_call(["git", "clone", "--depth", "1", "https://github.com/dscripka/openWakeWord.git", str(OPENWAKEWORD_DIR)])
else:
    print("openWakeWord ya existe en /content; se reutiliza.")

commit = subprocess.check_output(["git", "-C", str(OPENWAKEWORD_DIR), "rev-parse", "HEAD"], text=True).strip()
(DRIVE_ROOT / "openwakeword_commit.txt").write_text(commit + "
", encoding="utf-8")

py_paths = [str(OPENWAKEWORD_DIR), str(PIPER_DIR)]
os.environ["PYTHONPATH"] = os.pathsep.join(py_paths + [os.environ.get("PYTHONPATH", "")])
print("openWakeWord commit:", commit)
print("PYTHONPATH:", os.environ["PYTHONPATH"])


In [ ]:
# 5. Preparar Piper temporal en /content y copiar modelo persistente desde Drive
import shutil
import subprocess
from pathlib import Path

if not PIPER_DIR.exists():
    subprocess.check_call([
        "git", "clone", "--branch", "v2.0.0", "--depth", "1",
        "https://github.com/rhasspy/piper-sample-generator.git", str(PIPER_DIR)
    ])
else:
    print("piper-sample-generator temporal ya existe; se reutiliza.")

(PIPER_DIR / "models").mkdir(parents=True, exist_ok=True)

PIPER_PT_DRIVE = PIPER_MODELS_DRIVE / "en_US-libritts_r-medium.pt"
PIPER_JSON_DRIVE = PIPER_MODELS_DRIVE / "en_US-libritts_r-medium.pt.json"
PIPER_PT_TEMP = PIPER_DIR / "models" / "en_US-libritts_r-medium.pt"
PIPER_JSON_TEMP = PIPER_DIR / "models" / "en_US-libritts_r-medium.pt.json"

missing = [str(p) for p in [PIPER_PT_DRIVE, PIPER_JSON_DRIVE] if not p.exists() or p.stat().st_size == 0]
if missing:
    raise FileNotFoundError(
        "Falta el modelo Piper persistente en Drive. Colocalo antes de generar clips:
" + "
".join(missing)
    )

shutil.copy2(PIPER_PT_DRIVE, PIPER_PT_TEMP)
shutil.copy2(PIPER_JSON_DRIVE, PIPER_JSON_TEMP)
print("Piper temporal listo con modelo copiado desde Drive.")
print(PIPER_PT_TEMP)
print(PIPER_JSON_TEMP)


In [ ]:
# 6. Aplicar parches idempotentes de compatibilidad
from pathlib import Path
import importlib.util
import re
import site


def replace_once(path: Path, old: str, new: str, label: str) -> None:
    text = path.read_text(encoding="utf-8")
    if new in text:
        print(f"{label}: ya aplicado")
        return
    if old not in text:
        raise RuntimeError(f"No se encontro el patron esperado para {label} en {path}")
    path.write_text(text.replace(old, new, 1), encoding="utf-8")
    print(f"{label}: aplicado")

# Piper: torch.load(..., weights_only=False)
piper_generate = PIPER_DIR / "generate_samples.py"
text = piper_generate.read_text(encoding="utf-8")
updated = text.replace("torch.load(voice_path)", "torch.load(voice_path, weights_only=False)")
updated = updated.replace("torch.load(model_path)", "torch.load(model_path, weights_only=False)")
if updated != text:
    piper_generate.write_text(updated, encoding="utf-8")
    print("Piper torch.load: aplicado")
else:
    print("Piper torch.load: ya estaba correcto")

# deep-phonemizer: agregar weights_only=False solo donde falte dentro de torch.load(...)
def patch_deep_phonemizer() -> None:
    roots = []
    spec = importlib.util.find_spec("dp") or importlib.util.find_spec("deep_phonemizer")
    if spec and spec.submodule_search_locations:
        roots.extend(Path(p) for p in spec.submodule_search_locations)
    for site_dir in site.getsitepackages():
        candidate = Path(site_dir) / "dp"
        if candidate.exists():
            roots.append(candidate)
    patched = 0
    for root in sorted(set(roots)):
        for py_file in root.rglob("*.py"):
            source = py_file.read_text(encoding="utf-8")
            if "torch.load(" not in source:
                continue
            updated = re.sub(
                r"torch\.load\((checkpoint_path|model_path)(?!,\s*weights_only=)",
                r"torch.load(\1, weights_only=False",
                source,
            )
            if updated != source:
                py_file.write_text(updated, encoding="utf-8")
                patched += 1
    print(f"deep-phonemizer torch.load: archivos parcheados {patched}")

patch_deep_phonemizer()

# openWakeWord data.py: solo la llamada NumPy posterior a convertir el lote a NumPy.
data_py = OPENWAKEWORD_DIR / "openwakeword" / "data.py"
replace_once(
    data_py,
    "np.where(mixed_clips_batch.max(dim=1) != 0)[0]",
    "np.where(mixed_clips_batch.max(axis=1) != 0)[0]",
    "openWakeWord data.py max(axis=1)",
)

# openWakeWord train.py: exportar ONNX y saltar conversion TFLite.
train_py = OPENWAKEWORD_DIR / "openwakeword" / "train.py"
replace_once(
    train_py,
    "# Convert the model from onnx to tflite format
        convert_onnx_to_tflite(os.path.join(config["output_dir"], config["model_name"] + ".onnx"),
                               os.path.join(config["output_dir"], config["model_name"] + ".tflite"))",
    "# ORION patch: ONNX export only; skip TFLite conversion.
        logging.info("ORION patch: skipping TFLite conversion; ONNX export only.")",
    "openWakeWord train.py sin TFLite",
)


In [ ]:
# 7. Verificar recursos persistentes y descargar solo lo faltante
from pathlib import Path
import json
import shutil
import subprocess
import zipfile

FEATURE_URLS = {
    "openwakeword_features_ACAV100M_2000_hrs_16bit.npy": "https://huggingface.co/datasets/davidscripka/openwakeword_features/resolve/main/openwakeword_features_ACAV100M_2000_hrs_16bit.npy",
    "validation_set_features.npy": "https://huggingface.co/datasets/davidscripka/openwakeword_features/resolve/main/validation_set_features.npy",
}
MODEL_URLS = {
    "melspectrogram.onnx": "https://github.com/dscripka/openWakeWord/releases/download/v0.5.1/melspectrogram.onnx",
    "embedding_model.onnx": "https://github.com/dscripka/openWakeWord/releases/download/v0.5.1/embedding_model.onnx",
}
FMA_ZIP_URL = "https://os.unil.cloud.switch.ch/fma/fma_small.zip"
MIT_RIR_ZIP_URL = "https://mcdermottlab.mit.edu/Reverb/IRMAudio/Audio.zip"
ARCHIVES_DIR = DATASETS_DIR / "_archives"
ARCHIVES_DIR.mkdir(parents=True, exist_ok=True)


def require_command(name: str) -> None:
    if shutil.which(name) is None:
        raise RuntimeError(f"Falta {name}. En Colab deberia estar disponible antes de continuar.")


def download(url: str, target: Path) -> None:
    if target.exists() and target.stat().st_size > 0:
        print("OK existe:", target)
        return
    print("Descargando:", url)
    target.parent.mkdir(parents=True, exist_ok=True)
    subprocess.check_call(["wget", "-O", str(target), url])
    if not target.exists() or target.stat().st_size == 0:
        raise RuntimeError(f"Descarga vacia o fallida: {target}")


def convert_audio(src: Path, dst: Path) -> None:
    dst.parent.mkdir(parents=True, exist_ok=True)
    tmp = dst.with_suffix(".tmp.wav")
    subprocess.check_call([
        "ffmpeg", "-hide_banner", "-loglevel", "error", "-y",
        "-i", str(src), "-ac", "1", "-ar", "16000", "-sample_fmt", "s16", str(tmp)
    ])
    tmp.replace(dst)


def is_pcm16_mono_16k(path: Path) -> bool:
    try:
        raw = subprocess.check_output([
            "ffprobe", "-v", "error", "-select_streams", "a:0",
            "-show_entries", "stream=codec_name,sample_rate,channels",
            "-of", "json", str(path)
        ], text=True)
        stream = json.loads(raw).get("streams", [{}])[0]
        return stream.get("codec_name") == "pcm_s16le" and str(stream.get("sample_rate")) == "16000" and int(stream.get("channels", 0)) == 1
    except Exception:
        return False

require_command("wget")
require_command("ffmpeg")

resources_dir = OPENWAKEWORD_DIR / "openwakeword" / "resources" / "models"
resources_dir.mkdir(parents=True, exist_ok=True)
for name, url in MODEL_URLS.items():
    download(url, resources_dir / name)

for name, url in FEATURE_URLS.items():
    download(url, FEATURES_DIR / name)

# FMA: si ya hay al menos 120 WAV validos, no descargar ni convertir de nuevo.
fma_wavs = sorted(FMA_PATH.glob("*.wav"))
if len(fma_wavs) >= 120:
    print(f"FMA OK: {len(fma_wavs)} WAV existentes")
else:
    fma_zip = ARCHIVES_DIR / "fma_small.zip"
    download(FMA_ZIP_URL, fma_zip)
    extract_dir = ARCHIVES_DIR / "fma_small_extract"
    if not extract_dir.exists():
        with zipfile.ZipFile(fma_zip) as zf:
            zf.extractall(extract_dir)
    candidates = sorted([p for p in extract_dir.rglob("*") if p.suffix.lower() in {".mp3", ".wav", ".flac", ".ogg", ".m4a"}])[:120]
    if len(candidates) < 120:
        raise RuntimeError(f"FMA no contiene suficientes audios: {len(candidates)}")
    for index, src in enumerate(candidates, start=1):
        dst = FMA_PATH / f"fma_background_{index:03d}.wav"
        if not dst.exists() or dst.stat().st_size == 0:
            convert_audio(src, dst)
    print("FMA convertido a WAV mono 16 kHz PCM16:", len(list(FMA_PATH.glob("*.wav"))))

fma_converted = 0
for wav in sorted(FMA_PATH.glob("*.wav")):
    if not is_pcm16_mono_16k(wav):
        convert_audio(wav, wav)
        fma_converted += 1
print(f"FMA normalizacion adicional: {fma_converted} WAV")

# MIT RIR: si ya hay al menos 270 WAV, no descargar ni extraer de nuevo.
rir_wavs = sorted(MIT_RIRS_PATH.glob("*.wav"))
if len(rir_wavs) >= 270:
    print(f"MIT RIR OK: {len(rir_wavs)} WAV existentes")
else:
    rir_zip = ARCHIVES_DIR / "mit_rir_audio.zip"
    download(MIT_RIR_ZIP_URL, rir_zip)
    extract_dir = ARCHIVES_DIR / "mit_rirs_extract"
    if not extract_dir.exists():
        with zipfile.ZipFile(rir_zip) as zf:
            zf.extractall(extract_dir)
    candidates = sorted([p for p in extract_dir.rglob("*.wav")])[:270]
    if len(candidates) < 270:
        raise RuntimeError(f"MIT RIR no contiene suficientes WAV: {len(candidates)}")
    for index, src in enumerate(candidates, start=1):
        dst = MIT_RIRS_PATH / f"mit_rir_{index:03d}.wav"
        if not dst.exists() or dst.stat().st_size == 0:
            convert_audio(src, dst)
    print("MIT RIR listo:", len(list(MIT_RIRS_PATH.glob("*.wav"))))


In [ ]:
# 8. Crear/verificar my_model_v2.yaml
import yaml

config = {
    "target_phrase": ["oh ree on", "o ree on", "oree on", "orion"],
    "model_name": "orion_v2",
    "output_dir": str(OUTPUT_DIR),
    "n_samples": 1500,
    "n_samples_val": 1000,
    "steps": 10000,
    "target_accuracy": 0.6,
    "target_recall": 0.25,
    "piper_sample_generator_path": str(PIPER_DIR),
    "background_paths": [str(FMA_PATH), str(MIT_RIRS_PATH)],
    "background_paths_duplication_rate": [1, 1],
    "rir_paths": [str(MIT_RIRS_PATH)],
    "custom_negative_phrases": [
        "abre la calculadora", "enciende la luz", "que hora es", "abre el navegador",
        "hola asistente", "pon musica", "busca archivos", "cierra la ventana"
    ],
    "tts_batch_size": 50,
    "augmentation_rounds": 1,
    "augmentation_batch_size": 16,
    "feature_data_files": [str(FEATURES_DIR / "openwakeword_features_ACAV100M_2000_hrs_16bit.npy")],
    "false_positive_validation_data_path": str(FEATURES_DIR / "validation_set_features.npy"),
    "model_type": "dnn",
    "layer_size": 128,
    "batch_n_per_class": 32,
    "max_negative_weight": 1500,
    "target_false_positives_per_hour": 0.5,
}

existing = yaml.safe_load(CONFIG_PATH.read_text(encoding="utf-8")) if CONFIG_PATH.exists() else None
if existing == config:
    print("my_model_v2.yaml ya coincide con la configuracion v2.")
else:
    CONFIG_PATH.write_text(yaml.safe_dump(config, sort_keys=False, allow_unicode=False), encoding="utf-8")
    print("my_model_v2.yaml escrito/actualizado:", CONFIG_PATH)


In [ ]:
# 9. Diagnostico y normalizacion de los 5000 WAV generados
import json
import subprocess
from collections import Counter
from pathlib import Path

CLIP_DIRS = {
    "positive_train": 1500,
    "positive_test": 1000,
    "negative_train": 1500,
    "negative_test": 1000,
}


def ffprobe_info(path: Path) -> dict:
    raw = subprocess.check_output([
        "ffprobe", "-v", "error", "-select_streams", "a:0",
        "-show_entries", "stream=codec_name,sample_rate,channels,bits_per_sample",
        "-of", "json", str(path)
    ], text=True)
    streams = json.loads(raw).get("streams", [])
    if not streams:
        raise RuntimeError(f"Sin stream de audio: {path}")
    return streams[0]


def is_wav_pcm16_mono_16k(path: Path) -> bool:
    try:
        info = ffprobe_info(path)
    except Exception:
        return False
    return (
        info.get("codec_name") == "pcm_s16le"
        and str(info.get("sample_rate")) == "16000"
        and int(info.get("channels", 0)) == 1
    )


def normalize_wav(path: Path) -> None:
    tmp = path.with_name(path.stem + ".normalized.tmp.wav")
    subprocess.check_call([
        "ffmpeg", "-hide_banner", "-loglevel", "error", "-y",
        "-i", str(path), "-ac", "1", "-ar", "16000", "-sample_fmt", "s16", str(tmp)
    ])
    tmp.replace(path)

print("Estado Piper")
for p in [PIPER_PT_TEMP, PIPER_JSON_TEMP]:
    print(" ", p, "OK" if p.exists() and p.stat().st_size > 0 else "FALTA")

print("Estado datasets")
print(" FMA WAV:", len(list(FMA_PATH.glob("*.wav"))))
print(" MIT RIR WAV:", len(list(MIT_RIRS_PATH.glob("*.wav"))))

print("Conteos de clips y frecuencias encontradas")
all_rates = Counter()
for name, expected in CLIP_DIRS.items():
    directory = OUTPUT_DIR / name
    wavs = sorted(directory.glob("*.wav"))
    print(f" {name}: {len(wavs)} / minimo {expected}")
    if len(wavs) < expected:
        raise RuntimeError(f"Faltan WAV en {directory}. Ejecuta la generacion de clips antes de augment_clips.")
    converted = 0
    for wav in wavs:
        try:
            info = ffprobe_info(wav)
            all_rates[str(info.get("sample_rate"))] += 1
        except Exception:
            all_rates["invalid"] += 1
        if not is_wav_pcm16_mono_16k(wav):
            normalize_wav(wav)
            converted += 1
    print(f"   normalizados automaticamente: {converted}")

print("Frecuencias de muestreo encontradas antes/durante auditoria:", dict(all_rates))

print("Estado features negativas persistentes")
for p in [FEATURES_DIR / "openwakeword_features_ACAV100M_2000_hrs_16bit.npy", FEATURES_DIR / "validation_set_features.npy"]:
    print(" ", p.name, p.stat().st_size if p.exists() else "FALTA")

print("Estado de las cuatro features generadas")
for name in ["positive_features_train.npy", "positive_features_test.npy", "negative_features_train.npy", "negative_features_test.npy"]:
    p = OUTPUT_DIR / name
    print(" ", name, p.stat().st_size if p.exists() else "FALTA")

onnx_path = OUTPUT_DIR / "orion_v2.onnx"
export_path = EXPORTS_DIR / "orion_v2.onnx"
print("Estado del ONNX")
print(" training_output:", onnx_path.stat().st_size if onnx_path.exists() else "FALTA")
print(" exports:", export_path.stat().st_size if export_path.exists() else "FALTA")


In [ ]:
# 10. Generar features con log persistente
import os
import subprocess
import sys
import numpy as np
from pathlib import Path

FEATURE_FILES = [
    OUTPUT_DIR / "positive_features_train.npy",
    OUTPUT_DIR / "positive_features_test.npy",
    OUTPUT_DIR / "negative_features_train.npy",
    OUTPUT_DIR / "negative_features_test.npy",
]


def valid_npy(path: Path) -> bool:
    if not path.exists() or path.stat().st_size == 0:
        return False
    try:
        arr = np.load(path, mmap_mode="r")
        return getattr(arr, "size", 0) > 0
    except Exception:
        return False

for feature_file in FEATURE_FILES:
    if feature_file.exists() and not valid_npy(feature_file):
        print("Eliminando feature parcial o vacia:", feature_file)
        feature_file.unlink()

if all(valid_npy(p) for p in FEATURE_FILES):
    print("Las cuatro features ya estan completas; no se ejecuta --augment_clips.")
else:
    for name, expected in CLIP_DIRS.items():
        count = len(list((OUTPUT_DIR / name).glob("*.wav")))
        if count < expected:
            raise RuntimeError(f"No se ejecuta --augment_clips: {name} tiene {count}, esperado minimo {expected}.")
    cmd = [
        sys.executable, str(OPENWAKEWORD_DIR / "openwakeword" / "train.py"),
        "--training_config", str(CONFIG_PATH),
        "--augment_clips",
    ]
    env = os.environ.copy()
    env["PYTHONPATH"] = os.pathsep.join([str(OPENWAKEWORD_DIR), str(PIPER_DIR), env.get("PYTHONPATH", "")])
    print("Ejecutando:", " ".join(cmd))
    print("Log persistente:", AUGMENT_LOG)
    with AUGMENT_LOG.open("w", encoding="utf-8") as log:
        proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, env=env, bufsize=1)
        assert proc.stdout is not None
        for line in proc.stdout:
            print(line, end="")
            log.write(line)
        rc = proc.wait()
    if rc != 0:
        raise RuntimeError(f"--augment_clips fallo con codigo {rc}. Revisa {AUGMENT_LOG}")

missing = [str(p) for p in FEATURE_FILES if not valid_npy(p)]
if missing:
    raise RuntimeError("Features incompletas tras augment_clips:
" + "
".join(missing))
print("Features completas:")
for p in FEATURE_FILES:
    print(" ", p.name, p.stat().st_size, "bytes")


In [ ]:
# 11. Verificar features antes de entrenar
import numpy as np

for p in FEATURE_FILES:
    arr = np.load(p, mmap_mode="r")
    print(p.name, "shape=", arr.shape, "bytes=", p.stat().st_size)
    if arr.size == 0:
        raise RuntimeError(f"Feature vacia: {p}")
print("Las cuatro features estan listas para --train_model.")


In [ ]:
# 12. Entrenar ONNX, verificar export y descargar
import os
import shutil
import subprocess
import sys
from pathlib import Path

onnx_path = OUTPUT_DIR / "orion_v2.onnx"
export_path = EXPORTS_DIR / "orion_v2.onnx"

if not all(p.exists() and p.stat().st_size > 0 for p in FEATURE_FILES):
    raise RuntimeError("No se entrena: faltan features completas.")

if onnx_path.exists() and onnx_path.stat().st_size > 0:
    print("ONNX ya existe; no se reentrena:", onnx_path)
else:
    cmd = [
        sys.executable, str(OPENWAKEWORD_DIR / "openwakeword" / "train.py"),
        "--training_config", str(CONFIG_PATH),
        "--train_model",
    ]
    env = os.environ.copy()
    env["PYTHONPATH"] = os.pathsep.join([str(OPENWAKEWORD_DIR), str(PIPER_DIR), env.get("PYTHONPATH", "")])
    print("Ejecutando entrenamiento:", " ".join(cmd))
    print("Log persistente:", TRAIN_LOG)
    with TRAIN_LOG.open("w", encoding="utf-8") as log:
        proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, env=env, bufsize=1)
        assert proc.stdout is not None
        for line in proc.stdout:
            print(line, end="")
            log.write(line)
        rc = proc.wait()
    if rc != 0:
        raise RuntimeError(f"--train_model fallo con codigo {rc}. Revisa {TRAIN_LOG}")

if not onnx_path.exists() or onnx_path.stat().st_size == 0:
    raise RuntimeError(f"No se encontro ONNX valido en {onnx_path}")

EXPORTS_DIR.mkdir(parents=True, exist_ok=True)
shutil.copy2(onnx_path, export_path)
print("ONNX exportado a Drive:", export_path, export_path.stat().st_size, "bytes")

from google.colab import files
files.download(str(export_path))
